# 과제 1 · 연습문제 6.10~6.15

## 6.10 삭제할 노드를 알고 있을 때의 시간 복잡도

단순 연결 구조는 일반적으로 **O(n)**이다. 삭제할 노드를 알고 있어도 그 앞 노드의 링크를 수정해야 하므로, head부터 순서대로 탐색하여 선행 노드를 찾아야 한다. 단, 맨 앞 노드를 삭제하거나 선행 노드까지 알고 있다면 O(1)이다.

이중 연결 구조는 **O(1)**이다. 삭제할 노드의 prev로 선행 노드를 바로 알 수 있으므로, 앞뒤 노드의 링크만 수정하면 된다. 맨 앞이나 맨 뒤를 삭제할 때는 head 또는 tail도 갱신한다.

## 6.11 원형 연결 큐에서 tail을 사용하는 이유

원형 연결 구조에서는 마지막 노드의 링크가 첫 노드를 가리킨다. 따라서 tail을 알면 마지막 노드는 tail, 첫 노드는 tail.link로 바로 접근할 수 있다. enqueue는 tail 뒤에 삽입하고 tail을 갱신하며, dequeue는 tail.link를 제거하므로 둘 다 O(1)이다.

head만 저장하면 첫 노드는 바로 찾을 수 있지만, 마지막 노드를 찾으려면 리스트를 한 바퀴 순회해야 하므로 삽입에 O(n)이 걸린다. 따라서 tail 하나를 저장하는 것이 효율적이다.

## 공통 코드

교재 코드 6.5의 Node와 LinkedList를 기준으로 작성했다. 아래 셀부터 순서대로 실행한다.

In [1]:
class Node:
    def __init__(self, elem, link=None):
        self.data = elem
        self.link = link


class LinkedList:
    def __init__(self):
        self.head = None

    def isEmpty(self):
        return self.head is None

    def isFull(self):
        return False

    def getNode(self, pos):
        if pos < 0:
            return None
        node = self.head
        while pos > 0 and node is not None:
            node = node.link
            pos -= 1
        return node

    def getEntry(self, pos):
        node = self.getNode(pos)
        return None if node is None else node.data

    def insert(self, pos, elem):
        before = self.getNode(pos - 1)
        if before is None:
            self.head = Node(elem, self.head)
        else:
            before.link = Node(elem, before.link)

    def delete(self, pos):
        before = self.getNode(pos - 1)
        if before is None:
            if self.head is not None:
                self.head = self.head.link
        elif before.link is not None:
            before.link = before.link.link

    def size(self):
        count = 0
        node = self.head
        while node is not None:
            count += 1
            node = node.link
        return count

    def __str__(self):
        items = []
        node = self.head
        while node is not None:
            items.append(str(node.data))
            node = node.link
        return "[" + ", ".join(items) + "]"


## 6.12 printForward()

head부터 링크를 따라가며 데이터를 출력한다. 시간 복잡도는 O(n), 추가 공간은 O(1)이다.

아래 함수들을 LinkedList의 메서드로 등록하여 같은 클래스에 기능을 추가한다.

In [2]:
def printForward(self):
    node = self.head
    while node is not None:
        print(node.data, end=" ")
        node = node.link
    print()


LinkedList.printForward = printForward

A = LinkedList()
for i, value in enumerate([10, 20, 30, 40]):
    A.insert(i, value)
print("정순 출력:", end=" ")
A.printForward()

정순 출력: 10 20 30 40 


## 6.13 printReverse()

순환(재귀)을 이용해 다음 노드를 먼저 방문하고, 호출이 돌아오면서 현재 노드의 데이터를 출력한다. 링크는 변경하지 않는다. 시간 복잡도와 재귀 호출 공간은 O(n)이다.

In [3]:
def printReverse(self):
    def visit(node):
        if node is None:
            return
        visit(node.link)
        print(node.data, end=" ")

    visit(self.head)
    print()


LinkedList.printReverse = printReverse

print("역순 출력:", end=" ")
A.printReverse()
print("출력 후 원래 리스트:", A)

역순 출력: 40 30 20 10 
출력 후 원래 리스트: [10, 20, 30, 40]


## 6.14 append()

빈 리스트면 새 노드를 head로 지정하고, 비어 있지 않으면 마지막 노드를 찾아 새 노드를 연결한다. tail을 따로 저장하지 않으므로 시간 복잡도는 O(n)이다.

In [4]:
def append(self, elem):
    node = Node(elem)
    if self.head is None:
        self.head = node
    else:
        last = self.head
        while last.link is not None:
            last = last.link
        last.link = node


LinkedList.append = append

A.append(50)
print("50 추가 후:", A)
C = LinkedList()
C.append(100)
print("빈 리스트에 100 추가:", C)

50 추가 후: [10, 20, 30, 40, 50]
빈 리스트에 100 추가: [100]


## 6.15 merge()

A의 마지막 노드 뒤에 B의 시작 노드를 연결한 다음, B.head를 None으로 바꾼다. 노드를 복사하지 않고 이동하므로 병합 후 B의 길이는 0이 된다. A가 비어 있으면 B.head를 그대로 넘겨받는다. 시간 복잡도는 O(len(A))이며 추가 공간은 O(1)이다. 자기 자신과의 병합은 순환 연결이 생길 수 있어 허용하지 않는다.

In [5]:
def merge(self, B):
    if self is B:
        raise ValueError("자기 자신과는 병합할 수 없습니다.")
    if B.head is None:
        return
    if self.head is None:
        self.head = B.head
    else:
        last = self.head
        while last.link is not None:
            last = last.link
        last.link = B.head
    B.head = None


LinkedList.merge = merge

B = LinkedList()
for value in [60, 70, 80]:
    B.append(value)
print("병합 전 A:", A, "길이:", A.size())
print("병합 전 B:", B, "길이:", B.size())
A.merge(B)
print("병합 후 A:", A, "길이:", A.size())
print("병합 후 B:", B, "길이:", B.size())

병합 전 A: [10, 20, 30, 40, 50] 길이: 5
병합 전 B: [60, 70, 80] 길이: 3
병합 후 A: [10, 20, 30, 40, 50, 60, 70, 80] 길이: 8
병합 후 B: [] 길이: 0


## 실행 결과 확인

정순·역순 출력, 빈 리스트, 원소 한 개, 병합 후 길이와 노드 연결을 확인한다.

In [6]:
from contextlib import redirect_stdout
from io import StringIO


def make_list(values):
    result = LinkedList()
    for value in values:
        result.append(value)
    return result


def values_of(lst):
    result = []
    node = lst.head
    while node is not None:
        result.append(node.data)
        node = node.link
    return result


for values in [[], [1], [1, 2, 3]]:
    lst = make_list(values)
    for method, expected in [(lst.printForward, values),
                             (lst.printReverse, values[::-1])]:
        out = StringIO()
        with redirect_stdout(out):
            method()
        assert out.getvalue().split() == [str(x) for x in expected]
    assert values_of(lst) == values

for left, right in [([], []), ([], [1, 2]), ([1, 2], []),
                    ([1], [2]), ([1, 2], [3, 4])]:
    a, b = make_list(left), make_list(right)
    old_b_head = b.head
    a.merge(b)
    assert values_of(a) == left + right
    assert a.size() == len(left) + len(right)
    assert b.head is None and b.size() == 0
    if old_b_head is not None:
        assert a.getNode(len(left)) is old_b_head
    b.append(99)
    assert values_of(a) == left + right

single = make_list([7])
try:
    single.merge(single)
except ValueError:
    pass
else:
    raise AssertionError("자기 자신 병합 검사가 필요합니다.")
assert values_of(single) == [7]
print("전체 확인 완료: 정순·재귀 역순 출력, append, merge 정상")

전체 확인 완료: 정순·재귀 역순 출력, append, merge 정상
